# 01 — ETL Tablas de Dimensiones

Construye las 4 tablas de dimensiones del Star Schema a partir de los valores reales de los CSV raw.

**Output:** `data/clean/dim_tiempo.csv`, `dim_territorio.csv`, `dim_tipo_delito.csv`, `dim_demografia.csv`

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

BASE = Path(r'E:\informacion y documentos\Curso analisis de datos IT Academy\Proyecto final\criminalistica_cat')
RAW  = BASE / 'data' / 'raw'
CLEAN = BASE / 'data' / 'clean'
CLEAN.mkdir(parents=True, exist_ok=True)

print('BASE:', BASE)
print('CLEAN:', CLEAN)

---
## 1. dim_tiempo

Genera una fila por cada mes entre 2010 y 2025 (192 filas).

In [ ]:
NOMS_MES_CA = {
    1: 'gener', 2: 'febrer', 3: 'març', 4: 'abril',
    5: 'maig',  6: 'juny',  7: 'juliol', 8: 'agost',
    9: 'setembre', 10: 'octubre', 11: 'novembre', 12: 'desembre'
}

# Filas mensuales (para fact mensuales: Mossos, GUB)
filas = []
for anyo in range(2010, 2026):
    for mes in range(1, 13):
        trimestre = (mes - 1) // 3 + 1
        filas.append({
            'anyo': anyo,
            'mes': mes,
            'trimestre': trimestre,
            'nom_mes': NOMS_MES_CA[mes]
        })
dim_tiempo = pd.DataFrame(filas)

# Filas ANUALES (mes/trimestre = NULL) para fact_criminalidad_agregada (Ministerio/INE),
# que es anual. Convención: mes=<NA>, nom_mes='Any complet'. Se añaden DESPUÉS de las
# mensuales para que las mensuales conserven ids 1-192 (no rompe nb02/nb03).
filas_anuales = [
    {'anyo': anyo, 'mes': pd.NA, 'trimestre': pd.NA, 'nom_mes': 'Any complet'}
    for anyo in range(2010, 2026)
]
dim_tiempo = pd.concat([dim_tiempo, pd.DataFrame(filas_anuales)], ignore_index=True)
dim_tiempo['mes'] = dim_tiempo['mes'].astype('Int64')
dim_tiempo['trimestre'] = dim_tiempo['trimestre'].astype('Int64')

dim_tiempo.insert(0, 'id_tiempo', range(1, len(dim_tiempo) + 1))

n_mensuales = int(dim_tiempo['mes'].notna().sum())
n_anuales = int(dim_tiempo['mes'].isna().sum())
print(f'dim_tiempo: {len(dim_tiempo)} filas ({n_mensuales} mensuales + {n_anuales} anuales)')
print(dim_tiempo.head(3).to_string())
print('...')
print(dim_tiempo.tail(3).to_string())

In [ ]:
dim_tiempo.to_csv(CLEAN / 'dim_tiempo.csv', index=False, encoding='utf-8')
print('Guardado dim_tiempo.csv')

---
## 2. dim_territorio

Combina tres fuentes de territorio:
- **Mossos**: ABP + Regió Policial (nivel Cataluña)
- **GUB Barcelona**: barrios + distritos (nivel Barcelona ciudad)
- **Ministerio / INE**: CCAA + provincias (nivel nacional)

### 2a. Territorios Mossos (ABP)

In [ ]:
mossos = pd.read_csv(
    RAW / 'mossos' / 'mossos_fets_penals_raw.csv',
    encoding='utf-8',
    low_memory=False
)
print('Mossos shape:', mossos.shape)
mossos[['Regió Policial (RP)', 'Àrea Bàsica Policial (ABP)']].drop_duplicates().sort_values('Àrea Bàsica Policial (ABP)').head(10)

In [ ]:
territorios_mossos = (
    mossos[['Regió Policial (RP)', 'Àrea Bàsica Policial (ABP)']]
    .drop_duplicates()
    .rename(columns={
        'Regió Policial (RP)': 'region_policial',
        'Àrea Bàsica Policial (ABP)': 'abp'
    })
)
territorios_mossos['municipio'] = None
territorios_mossos['provincia'] = 'Cataluña'
territorios_mossos['ccaa'] = 'Cataluña'
territorios_mossos['cod_barri'] = None
territorios_mossos['nom_barri'] = None
territorios_mossos['cod_districte'] = None
territorios_mossos['nom_districte'] = None
territorios_mossos['nivel_territorial'] = 'abp'   # nivel sub-provincial (Àrea Bàsica Policial)
territorios_mossos['fuente'] = 'mossos'

print(f'Territorios Mossos: {len(territorios_mossos)} ABPs únicos')
territorios_mossos.head()

### 2b. Territorios GUB Barcelona (barrios + distritos)

In [ ]:
RENAME_GUB = {
    'Codi Incident': 'Codi_Incident',
    'Descripció Incident': 'Descripcio_Incident',
    'Codi districte': 'Codi_districte',
    'Nom districte': 'Nom_districte',
    'Codi barri': 'Codi_barri',
    'Nom barri': 'Nom_barri',
    'NK Any': 'NK_Any',
    'Mes de any': 'Mes_any',
    'Nom mes': 'Nom_mes',
    "Número d'incidents GUB": 'Numero_incidents_GUB'
}

gub_barrios_list = []
gub_path = RAW / 'gub_barcelona'

for f in sorted(gub_path.glob('*.csv')):
    df = pd.read_csv(f, encoding='utf-8', low_memory=False)
    df = df.rename(columns=RENAME_GUB)
    # Eliminar filas sin barrio/distrito (P5)
    df = df.dropna(subset=['Codi_districte'])
    # Normalizar códigos a int (vienen como float si el fichero tenía nulls)
    df['Codi_barri'] = df['Codi_barri'].astype(int)
    df['Codi_districte'] = df['Codi_districte'].astype(int)
    cols = ['Codi_barri', 'Nom_barri', 'Codi_districte', 'Nom_districte']
    gub_barrios_list.append(df[cols].drop_duplicates())

# Dedup por (Codi_barri, Codi_districte): el mismo barrio puede tener variantes
# de grafía en el nombre entre años (p.ej. 'el Poble Sec' vs 'el Poble-sec').
# Se mantiene la primera aparición para no duplicar territorios.
gub_barrios = (
    pd.concat(gub_barrios_list)
    .drop_duplicates(subset=['Codi_barri', 'Codi_districte'])
    .reset_index(drop=True)
)
print(f'Barrios GUB únicos: {len(gub_barrios)}')
gub_barrios.head()

In [ ]:
territorios_gub = gub_barrios.copy()
territorios_gub['abp'] = None
territorios_gub['region_policial'] = None
territorios_gub['municipio'] = 'Barcelona'
territorios_gub['provincia'] = 'Barcelona'
territorios_gub['ccaa'] = 'Cataluña'
territorios_gub['nivel_territorial'] = 'barri'   # nivel barrio (dentro de Barcelona ciudad)
territorios_gub['fuente'] = 'gub'

territorios_gub = territorios_gub.rename(columns={
    'Codi_barri': 'cod_barri',
    'Nom_barri': 'nom_barri',
    'Codi_districte': 'cod_districte',
    'Nom_districte': 'nom_districte'
})

print(f'Territorios GUB: {len(territorios_gub)} filas')

### 2c. Territorios Ministerio / INE (CCAA + provincias)

In [ ]:
min_cat = pd.read_csv(
    RAW / 'ministerio' / 'ministerio_hechos_conocidos_cataluna_filtrado.csv',
    encoding='utf-8-sig', sep=';'
)
ccaa_vals = min_cat['Comunidades autónomas'].dropna().unique()
print('CCAA Ministerio:', ccaa_vals)

portal = pd.read_csv(
    RAW / 'ministerio' / 'PORTAL ESTADÍSTICO DE CRIMINALIDAD Detenciones e investigados por tipología penal, grupo de edad y sexo.csv',
    encoding='utf-8-sig', sep=';', low_memory=False
)
print('\nTerritorios Portal (muestra):')
print(portal['Comunidades autónomas'].unique()[:10])

In [ ]:
# Los ficheros del Ministerio/INE usan grafías distintas para el mismo territorio:
#   'CATALUÑA' (Ministerio, Portal), '09 Cataluña' / 'Cataluña' (INE), 'Barcelona' (provincia)
# En lugar de volcar esas cadenas crudas (que crearía filas duplicadas para Cataluña y
# clasificaría mal provincia/ccaa), se construye un conjunto CANÓNICO de territorios y
# nb04 normaliza cada fuente a estos ids. Solo se modelan los relevantes para el proyecto
# (Cataluña + sus 4 provincias). Las demás CCAA y 'Total Nacional' no se incluyen.
print('Territorios canónicos a crear: Cataluña (CCAA) + Barcelona/Girona/Lleida/Tarragona (provincias)')

In [ ]:
# Conjunto canónico Ministerio/INE.
# - Cataluña: nivel CCAA (provincia = NaN, ccaa = 'Cataluña')
# - Barcelona/Girona/Lleida/Tarragona: nivel provincia (provincia = nombre, ccaa = 'Cataluña')
# nb04 distingue el nivel CCAA (provincia NaN) del provincial al hacer el lookup.
territorios_min = pd.DataFrame([
    {'provincia': pd.NA,       'ccaa': 'Cataluña'},   # nivel CCAA
    {'provincia': 'Barcelona', 'ccaa': 'Cataluña'},
    {'provincia': 'Girona',    'ccaa': 'Cataluña'},
    {'provincia': 'Lleida',    'ccaa': 'Cataluña'},
    {'provincia': 'Tarragona', 'ccaa': 'Cataluña'},
])
for col in ['cod_barri', 'nom_barri', 'cod_districte', 'nom_districte', 'municipio', 'abp', 'region_policial']:
    territorios_min[col] = None
# nivel: provincia NaN → 'ccaa' (toda Cataluña); con provincia → 'provincia'
territorios_min['nivel_territorial'] = territorios_min['provincia'].apply(
    lambda p: 'ccaa' if pd.isna(p) else 'provincia'
)
territorios_min['fuente'] = 'ministerio_ine'

print(f'Territorios canónicos Ministerio/INE: {len(territorios_min)}')
territorios_min[['provincia', 'ccaa', 'nivel_territorial', 'fuente']]

### 2d. Unir y asignar IDs

In [ ]:
COLS_TERRITORIO = [
    'cod_barri', 'nom_barri', 'cod_districte', 'nom_districte',
    'municipio', 'provincia', 'ccaa', 'abp', 'region_policial',
    'nivel_territorial', 'fuente'
]

# Alinear columnas
for df in [territorios_mossos, territorios_gub, territorios_min]:
    for col in COLS_TERRITORIO:
        if col not in df.columns:
            df[col] = None

dim_territorio = pd.concat(
    [territorios_mossos[COLS_TERRITORIO],
     territorios_gub[COLS_TERRITORIO],
     territorios_min[COLS_TERRITORIO]],
    ignore_index=True
).drop_duplicates()

dim_territorio.insert(0, 'id_territorio', range(1, len(dim_territorio) + 1))

print(f'dim_territorio: {len(dim_territorio)} filas')
print('\nDistribución por nivel_territorial:')
print(dim_territorio['nivel_territorial'].value_counts())
print('\nDistribución por fuente:')
print(dim_territorio['fuente'].value_counts())
dim_territorio.head()

In [ ]:
dim_territorio.to_csv(CLEAN / 'dim_territorio.csv', index=False, encoding='utf-8')
print('Guardado dim_territorio.csv')

---
## 3. dim_tipo_delito

Combina categorías de delito de tres fuentes:
- **Mossos**: `Títol Codi Penal` + `Tipus de fet`
- **GUB Barcelona**: `Codi_Incident` + `Descripcio_Incident`
- **Ministerio / INE**: `Tipología penal`

### 3a. Tipos Mossos

In [ ]:
tipos_mossos = (
    mossos[['Títol Codi Penal', 'Tipus de fet']]
    .drop_duplicates()
    .rename(columns={
        'Títol Codi Penal': 'titol_cp',
        'Tipus de fet': 'descripcio'
    })
)
tipos_mossos['codigo'] = None
tipos_mossos['categoria'] = tipos_mossos['titol_cp']
# Cada 'Tipus de fet' de Mossos es una hoja (no hay filas TOTAL/subtotal embebidas)
# → 'detalle': es seguro sumar todas las tipologías de Mossos sin doble conteo.
tipos_mossos['nivel_tipologia'] = 'detalle'
tipos_mossos['fuente'] = 'mossos'

print(f'Tipos Mossos: {len(tipos_mossos)}')
tipos_mossos.head()

### 3b. Tipos GUB

In [ ]:
gub_tipos_list = []
for f in sorted(gub_path.glob('*.csv')):
    df = pd.read_csv(f, encoding='utf-8', low_memory=False)
    df = df.rename(columns=RENAME_GUB)
    df['Descripcio_Incident'] = df['Descripcio_Incident'].str.strip()
    # Codi_Incident es alfanumérico (existe '21M') y se lee como int en unos
    # ficheros y str en otros → forzar str para que drop_duplicates no trate
    # 210 (int) y '210' (str) como categorías distintas.
    df['Codi_Incident'] = df['Codi_Incident'].astype(str)
    # Quitar la fila basura con Descripcio nula
    df = df.dropna(subset=['Descripcio_Incident'])
    cols = ['Codi_Incident', 'Descripcio_Incident']
    gub_tipos_list.append(df[cols].drop_duplicates())

tipos_gub = (
    pd.concat(gub_tipos_list)
    .drop_duplicates()
    .rename(columns={
        'Codi_Incident': 'codigo',
        'Descripcio_Incident': 'descripcio'
    })
)
tipos_gub['titol_cp'] = None
tipos_gub['categoria'] = None
# Cada incidente GUB es una hoja (sin totales embebidos) → 'detalle'
tipos_gub['nivel_tipologia'] = 'detalle'
tipos_gub['fuente'] = 'gub'

print(f'Tipos GUB: {len(tipos_gub)} categorías únicas')
tipos_gub.head()

### 3c. Tipos Ministerio / INE

In [ ]:
import re

tipos_min_vals = set()
min_tipologia_files = [
    ('ministerio_hechos_conocidos_cataluna_filtrado.csv', 'utf-8-sig', ';'),
    ('ministerio_detenciones_investigados_cataluna_filtrado.csv', 'utf-8-sig', ';'),
]
for fname, enc, sep in min_tipologia_files:
    df_tmp = pd.read_csv(RAW / 'ministerio' / fname, encoding=enc, sep=sep)
    if 'Tipología penal' in df_tmp.columns:
        tipos_min_vals.update(df_tmp['Tipología penal'].dropna().unique())

# Portal también tiene tipología
tipos_min_vals.update(portal['Tipología penal'].dropna().unique())

# INE delitos tipo — columnas reales: 'Tipo de Delito: Nivel 1/2/3/4'
ine_tipo = pd.read_csv(
    RAW / 'ine' / 'ine_ccaa_delitos_tipo_raw.csv',
    encoding='utf-8', sep=';'
)
print('Columnas INE tipo:', ine_tipo.columns.tolist())
cols_tipo_ine = [c for c in ine_tipo.columns if c.startswith('Tipo de Delito')]
for col in cols_tipo_ine:
    tipos_min_vals.update(ine_tipo[col].dropna().unique())


def nivel_tipologia_ministerio(desc):
    """Deriva el nivel jerárquico de la tipología penal del Ministerio/Portal a partir
    del código numérico que la precede:
      'TOTAL INFRACCIONES PENALES'        -> 'total'        (suma de todo)
      '5. PATRIMONIO'                     -> 'categoria'    (1 segmento)
      '5.2.-Robos con fuerza...'          -> 'subcategoria' (2 segmentos)
      '5.2.1.-Robos ... vehículos'        -> 'detalle'      (3+ segmentos)
    Los nombres sin código (taxonomía INE delitos-tipo) se marcan 'detalle'.
    REGLA DE ANÁLISIS: para sumar sin doble conteo, usar un único nivel
    (o excluir 'total' y agregados superiores)."""
    d = str(desc).strip()
    if d.upper().startswith('TOTAL'):
        return 'total'
    m = re.match(r'^(\d+(?:\.\d+)*)', d)
    if m:
        n_seg = len(m.group(1).split('.'))
        return {1: 'categoria', 2: 'subcategoria'}.get(n_seg, 'detalle')
    return 'detalle'  # nombres sin código numérico (INE)


tipos_ministerio = pd.DataFrame({
    'descripcio': sorted(tipos_min_vals),
    'codigo': None,
    'titol_cp': None,
    'categoria': None,
    'fuente': 'ministerio_ine'
})
tipos_ministerio['nivel_tipologia'] = tipos_ministerio['descripcio'].map(nivel_tipologia_ministerio)

print(f'\nTipos Ministerio/INE: {len(tipos_ministerio)}')
print('Distribución por nivel_tipologia:')
print(tipos_ministerio['nivel_tipologia'].value_counts())
tipos_ministerio.head(10)

### 3d. Unir y asignar IDs

In [ ]:
COLS_DELITO = ['codigo', 'descripcio', 'titol_cp', 'categoria', 'nivel_tipologia', 'fuente']

dim_tipo_delito = pd.concat(
    [tipos_mossos[COLS_DELITO],
     tipos_gub[COLS_DELITO],
     tipos_ministerio[COLS_DELITO]],
    ignore_index=True
).drop_duplicates()

dim_tipo_delito.insert(0, 'id_tipo_delito', range(1, len(dim_tipo_delito) + 1))

print(f'dim_tipo_delito: {len(dim_tipo_delito)} filas')
print('\nDistribución por nivel_tipologia:')
print(dim_tipo_delito['nivel_tipologia'].value_counts())
print('\nDistribución por fuente:')
print(dim_tipo_delito['fuente'].value_counts())
dim_tipo_delito.head()

In [ ]:
dim_tipo_delito.to_csv(CLEAN / 'dim_tipo_delito.csv', index=False, encoding='utf-8')
print('Guardado dim_tipo_delito.csv')

---
## 4. dim_demografia

Combina valores de sexo, grupo de edad y nacionalidad de:
- Portal Ministerio (detenciones edad+sexo)
- INE condenados (sexo+edad, sexo+nacionalidad)
- Idescat penitenciaria (sexo, edad, nacionalidad)

### 4a. Valores Portal Ministerio

In [ ]:
print('Columnas Portal:')
print(portal.columns.tolist())
print('\nGrupo de edad únicos:')
print(sorted(portal['Grupo de edad'].dropna().unique()))
print('\nSexo únicos:')
print(sorted(portal['Sexo'].dropna().unique()))

In [ ]:
demo_portal = (
    portal[['Grupo de edad', 'Sexo']]
    .drop_duplicates()
    .rename(columns={'Grupo de edad': 'grup_edat', 'Sexo': 'sexo'})
)
demo_portal['nacionalitat'] = None
demo_portal['tipus_nacionalitat'] = None
demo_portal['fuente'] = 'portal_ministerio'

print(f'Demo Portal: {len(demo_portal)} combinaciones')
demo_portal

### 4b. Valores INE condenados

In [ ]:
ine_edad = pd.read_csv(
    RAW / 'ine' / 'ine_ccaa_condenados_sexo_edad_raw.csv',
    encoding='utf-8', sep=';'
)
ine_edad['Edad'] = ine_edad['Edad'].str.strip()  # P8: trailing space

demo_ine_edad = (
    ine_edad[['Sexo', 'Edad']]
    .drop_duplicates()
    .rename(columns={'Sexo': 'sexo', 'Edad': 'grup_edat'})
)
demo_ine_edad['nacionalitat'] = None
demo_ine_edad['tipus_nacionalitat'] = None
demo_ine_edad['fuente'] = 'ine_condenados'

print('Edades INE:', sorted(demo_ine_edad['grup_edat'].unique()))

In [ ]:
ine_nac = pd.read_csv(
    RAW / 'ine' / 'ine_ccaa_condenados_sexo_nacionalidad_raw.csv',
    encoding='utf-8', sep=';'
)
print('Columnas INE nacionalidad:', ine_nac.columns.tolist())
print('\nNacionalidad únicos:', sorted(ine_nac.iloc[:, 2].dropna().unique())[:10])

In [ ]:
col_nac = [c for c in ine_nac.columns if 'nacion' in c.lower() or 'Nacion' in c][0]
col_sexo = [c for c in ine_nac.columns if 'Sexo' in c or 'sexo' in c][0]

demo_ine_nac = (
    ine_nac[[col_sexo, col_nac]]
    .drop_duplicates()
    .rename(columns={col_sexo: 'sexo', col_nac: 'nacionalitat'})
)
demo_ine_nac['grup_edat'] = None
demo_ine_nac['tipus_nacionalitat'] = None
demo_ine_nac['fuente'] = 'ine_condenados'

print(f'Demo INE nacionalidad: {len(demo_ine_nac)} combinaciones')

### 4c. Valores Idescat penitenciaria

In [ ]:
# Usar el fichero histórico limpio (el _raw tiene cabeceras de metadatos)
idescat_nac = pd.read_csv(
    RAW / 'idescat' / 'idescat_poblacion_penitenciaria_nacionalidad_historico_2010_2023.csv',
    encoding='utf-8-sig'
)
print('Idescat nac cols:', idescat_nac.columns.tolist())
idescat_nac.head(3)

In [ ]:
col_nac_ids = [c for c in idescat_nac.columns if 'nacion' in c.lower() or 'Nacion' in c.lower() or 'nacionalitat' in c.lower()]
print('Columnas nacionalitat:', col_nac_ids)

if col_nac_ids:
    demo_idescat = pd.DataFrame({
        'sexo': None,
        'grup_edat': None,
        'nacionalitat': idescat_nac[col_nac_ids[0]].dropna().unique(),
        'tipus_nacionalitat': None,
        'fuente': 'idescat'
    })
else:
    # Si no hay columna directa, mirar todas las columnas
    print('Todas las columnas:', idescat_nac.columns.tolist())
    demo_idescat = pd.DataFrame(columns=['sexo', 'grup_edat', 'nacionalitat', 'tipus_nacionalitat', 'fuente'])

print(f'\nDemo Idescat: {len(demo_idescat)} registros')

In [ ]:
# Usar el fichero histórico limpio (el _raw tiene cabeceras de metadatos)
idescat_sexo_edad = pd.read_csv(
    RAW / 'idescat' / 'idescat_poblacion_penitenciaria_sexo_edad_tipo_historico_2010_2023.csv',
    encoding='utf-8-sig'
)
print('Idescat sexo+edad cols:', idescat_sexo_edad.columns.tolist())
idescat_sexo_edad.head(3)

In [ ]:
# Extraer grupos de edad de las columnas (Edat_18_20, Edat_21_25, etc.)
cols_edad = [c for c in idescat_sexo_edad.columns if c.startswith('Edat') or 'edat' in c.lower() or 'edad' in c.lower()]
print('Columnas edad Idescat:', cols_edad)

grupos_idescat = pd.DataFrame({
    'sexo': None,
    'grup_edat': cols_edad,
    'nacionalitat': None,
    'tipus_nacionalitat': None,
    'fuente': 'idescat'
})
print(f'Grupos edad Idescat: {len(grupos_idescat)}')

### 4d. Unir y asignar IDs

In [ ]:
COLS_DEMO = ['sexo', 'grup_edat', 'nacionalitat', 'tipus_nacionalitat', 'fuente']

dfs_demo = [demo_portal, demo_ine_edad, demo_ine_nac, grupos_idescat]
if len(demo_idescat) > 0:
    dfs_demo.append(demo_idescat)

for df in dfs_demo:
    for col in COLS_DEMO:
        if col not in df.columns:
            df[col] = None

dim_demografia = pd.concat(
    [df[COLS_DEMO] for df in dfs_demo],
    ignore_index=True
).drop_duplicates()

dim_demografia.insert(0, 'id_demografia', range(1, len(dim_demografia) + 1))

print(f'dim_demografia: {len(dim_demografia)} filas')
print('\nDistribución por fuente:')
print(dim_demografia['fuente'].value_counts())
dim_demografia.head(10)

In [ ]:
dim_demografia.to_csv(CLEAN / 'dim_demografia.csv', index=False, encoding='utf-8')
print('Guardado dim_demografia.csv')

---
## 5. Resumen final

In [ ]:
print('=' * 50)
print('RESUMEN ETL DIMENSIONES')
print('=' * 50)

dimensiones = {
    'dim_tiempo': dim_tiempo,
    'dim_territorio': dim_territorio,
    'dim_tipo_delito': dim_tipo_delito,
    'dim_demografia': dim_demografia
}

for nombre, df in dimensiones.items():
    ruta = CLEAN / f'{nombre}.csv'
    existe = 'OK' if ruta.exists() else 'FALTA'
    print(f'[{existe}] {nombre}: {len(df)} filas -> {ruta.name}')

print('\nListo para continuar con 02_etl_mossos.ipynb')